# This script is used to test an OpenAI model

In [15]:
import os
from openai import OpenAI
from datetime import datetime
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
import json
import sys
sys.path.append('..')
import helper
from utils import clean_response, parse_response
import numpy as np
from datetime import datetime
import glob

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Load OPENAI_API_KEY from .env file
load_dotenv()

client = OpenAI()

In [3]:
TEST_PROMPTS = "../../prompts/domain_promts.json"
MODEL_NAME = "gpt-4.1-mini-2025-04-14"
OPENAI_TEST_FILE_ID = "file-Q5mRbt7ejvoDEPLkJ8hVKN"
TEST_SET_PATH = "/ceph/aasteine/fine-tuning-paper/data/wdc/wdcproducts80cc20rnd050un_test_gs.pkl"

In [4]:
def insert_product_descriptions(prompt_template: str, product1: str, product2: str):
    # Replace placeholder texts with actual product descriptions
    prompt = prompt_template.replace("'Entity 1'", product1).replace("'Entity 2'", product2)
    return prompt

In [5]:
def create_prompt(prompt, custom_id, model, product_1=None, product_2=None):
    if product_1 is not None and product_2 is not None:
        prompt = insert_product_descriptions(prompt, product_1, product_2)
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model,
            "messages": [
                {"role": "user", "content": prompt},
            ],
            "max_tokens": 5,
            "temperature": 0
        }
    }

In [6]:
def create_batch_job(test_dir:str, test_set_path:str, test_prompts_path:str):
    # Load the test set
    test_set = pd.read_pickle(test_set_path)

    # open fine-tune run config
    with open(os.path.join(test_dir, "fine-tune-run_config.json"), "r") as f:
        run_config = json.load(f)
        
    # get the name of the fine-tuned model via the openai api
    fine_tune_model = client.fine_tuning.jobs.retrieve(run_config['job_id']).fine_tuned_model

    # Create output directory structure
    run_name = run_config['run_name']
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Load all prompts we want to test
    with open(test_prompts_path, 'r') as file:
        prompts = json.load(file)

    batch_job = []
    print(f"Creating batch job for {run_name}. The batch will contain {len(prompts)} prompts. With {len(test_set)} pairs each.")
    for task in prompts:
        prompt_template = task['prompt']
        prompt_id = task['id']

        for index, row in test_set.iterrows():
            product1, product2 = row['title_left'], row['title_right']
            label = row.get('label') 
            pair_id = row['pair_id']
            
            custom_id = f"{prompt_id};{pair_id};{label}"
            prompt = create_prompt(prompt_template, custom_id, fine_tune_model, product1, product2)
            batch_job.append(prompt)
            

    # Save the input file
    input_file_path = os.path.join(test_dir, f"testing_{run_name}_input.jsonl")
    with open(input_file_path, "w") as f:
        for request in batch_job:
            f.write(json.dumps(request) + "\n")
            
    # upload the batch file to openai
    batch_input_file = client.files.create(
        file=open(input_file_path, "rb"),
        purpose="batch"
    )

    # create the batch
    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"description": run_name}
    )

    # Save the run configuration
    run_config = {
        "run_name": run_name,
        "timestamp": timestamp,
        "model": fine_tune_model,
        "batch_id": batch.id,
        "input_file_id": batch_input_file.id,
        "endpoint": "/v1/chat/completions",
        "completion_window": "24h",
        "metadata": batch.metadata,
        "created_at": batch.created_at,
        "status": batch.status
    }

    with open(os.path.join(test_dir, "test_run_config.json"), "w") as f:
        json.dump(run_config, f, indent=2)

    print(f"Run configuration and input files saved to: {test_dir}")
    print(f"Batch ID: {batch.id}")


In [7]:
def get_batch_results(testing_dir):
    with open(os.path.join(testing_dir, "test_run_config.json"), "r") as f:
        run_config = json.load(f)
    
    batch_id = run_config['batch_id']

    # Retrieve the batch
    batch = client.batches.retrieve(batch_id)
    print(f"Batch status: {batch.status}")
    print(f"Created at: {datetime.utcfromtimestamp(batch.created_at).strftime('%Y-%m-%d %H:%M:%S')}")

    # Check if the batch has completed and has an output file
    if batch.output_file_id:
        # Download the output file content
        file_content = client.files.content(batch.output_file_id)
        
        # Write the content to a .jsonl file
        output_file = os.path.join(testing_dir, f"batch_output_{batch_id}.jsonl")
        with open(output_file, "w") as f:
            f.write(file_content.text)
        
        print(f"Batch results saved to: {output_file}")
    else:
        print("Batch is not completed or output file is not available.")
    return output_file

In [8]:
def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    
    return accuracy, f1, precision, recall

In [9]:
def get_batch_results_and_calculate_metrics(testing_dir):
    # Load the result file 
    result_path = get_batch_results(testing_dir)
    gpt_result = pd.read_json(result_path, lines=True)


    # Split the custom_id into dataset, task, pair_id, and label
    gpt_result[['task', 'pair_id', 'label']] = gpt_result.custom_id.str.split(";", expand=True)
    gpt_result = gpt_result.drop(columns=['custom_id'])

    # Apply the parse_response function to the response column
    parsed_df = gpt_result["response"].apply(parse_response)

    # Concatenate the parsed results with the original DataFrame
    gpt_result = pd.concat([gpt_result, parsed_df], axis=1)

    # Transform 'content' to binary (0 or 1 based on "Yes")
    gpt_result['content'] = gpt_result['content'].apply(clean_response)

    # Convert label from string to integer
    gpt_result['label'] = gpt_result['label'].astype(int)

    # Group by 'dataset' and 'task', then calculate metrics
    results = []
    grouped = gpt_result.groupby(['task'])

    for (task), group in grouped:
        y_true = group['label']
        y_pred = group['content']
        
        accuracy, f1, precision, recall = calculate_metrics(y_true, y_pred)
        
        results.append({
            'task': task,
            'accuracy': accuracy,
            'f1_score': f1,
            'precision': precision,
            'recall': recall
        })

    # Convert the results into a DataFrame
    metrics_df = pd.DataFrame(results)

    # Save metrics to CSV in the same directory
    output_path = os.path.join(testing_dir, 'testing_stats.csv')
    metrics_df.to_csv(output_path, index=False)
    print(f"Metrics saved to {output_path}")
    print(f"Best performing task: {metrics_df.sort_values(by='f1_score', ascending=False).iloc[0]['f1_score']}")
    return output_path

## Baseline

In [14]:
# Load the test set
test_set = pd.read_pickle("/ceph/aasteine/fine-tuning-paper/data/wdc/wdcproducts80cc20rnd050un_test_gs.pkl")

# Create output directory structure
run_name = "baseline-wdc"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../../results/{MODEL_NAME}/{run_name}/{timestamp}"
os.makedirs(output_dir, exist_ok=True)

# Load all prompts we want to test
with open(TEST_PROMPTS, 'r') as file:
    prompts = json.load(file)

batch_job = []
print(f"Creating batch job for {run_name}. The batch will contain {len(prompts)} prompts. With {len(test_set)} pairs each.")
for task in prompts:
    title = task['title']
    prompt_template = task['prompt']
    prompt_id = task['id']

    for index, row in test_set.iterrows():
        product1, product2 = row['title_left'], row['title_right']
        label = row.get('label') 
        pair_id = row['pair_id']
        
        custom_id = f"{prompt_id};{pair_id};{label}"
        prompt = create_prompt(prompt_template, custom_id, MODEL_NAME, product1, product2)
        batch_job.append(prompt)
        

# Save the input file
input_file_path = os.path.join(output_dir, "input.jsonl")
with open(input_file_path, "w") as f:
    for request in batch_job:
        f.write(json.dumps(request) + "\n")
        
# upload the batch file to openai
batch_input_file = client.files.create(
    file=open(input_file_path, "rb"),
    purpose="batch"
)

# create the batch
batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={"description": run_name}
)

# Save the run configuration
run_config = {
    "run_name": run_name,
    "timestamp": timestamp,
    "model": MODEL_NAME,
    "batch_id": batch.id,
    "input_file_id": batch_input_file.id,
    "endpoint": "/v1/chat/completions",
    "completion_window": "24h",
    "metadata": batch.metadata,
    "created_at": batch.created_at,
    "status": batch.status
}

with open(os.path.join(output_dir, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print(f"Run configuration and input files saved to: {output_dir}")
print(f"Batch ID: {batch.id}")


Creating batch job for baseline-wdc. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/baseline-wdc/20250506_164602
Batch ID: batch_681a20b059408190a6fdd648305dc902


In [7]:
output_dir = "../../results/gpt-4.1-mini-2025-04-14/baseline-wdc/20250506_164602"

In [13]:
# Load the result file 
output_file = os.path.join(output_dir, f"batch_output_{batch_id}.jsonl")

gpt_result = pd.read_json(output_file, lines=True)


# Split the custom_id into dataset, task, pair_id, and label
gpt_result[['task', 'pair_id', 'label']] = gpt_result.custom_id.str.split(";", expand=True)
gpt_result = gpt_result.drop(columns=['custom_id'])

# Apply the parse_response function to the response column
parsed_df = gpt_result["response"].apply(parse_response)

# Concatenate the parsed results with the original DataFrame
gpt_result = pd.concat([gpt_result, parsed_df], axis=1)

# Transform 'content' to binary (0 or 1 based on "Yes")
gpt_result['content'] = gpt_result['content'].apply(lambda x: 1 if "Yes" in x else 0)

# Convert label from string to integer
gpt_result['label'] = gpt_result['label'].astype(int)

# Group by 'dataset' and 'task', then calculate metrics
results = []
grouped = gpt_result.groupby(['task'])

for (task), group in grouped:
    y_true = group['label']
    y_pred = group['content']
    
    accuracy, f1, precision, recall = calculate_metrics(y_true, y_pred)
    
    results.append({
        'task': task,
        'accuracy': accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    })

# Convert the results into a DataFrame
metrics_df = pd.DataFrame(results)

# Save metrics to CSV in the same directory
output_path = os.path.join(output_dir, 'stats.csv')
metrics_df.to_csv(output_path, index=False)
print(f"Metrics saved to {output_path}")

Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/baseline-wdc/20250506_164602/stats.csv


## WDC regular fine-tuning

In [8]:
test_dir = "../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809"

# Load the test set
test_set = pd.read_pickle("/ceph/aasteine/fine-tuning-paper/data/wdc/wdcproducts80cc20rnd050un_test_gs.pkl")


# open fine-tune run config
with open(os.path.join(test_dir, "fine-tune-run_config.json"), "r") as f:
    run_config = json.load(f)
    
# get the name of the fine-tuned model via the openai api
fine_tune_model = client.fine_tuning.jobs.retrieve(run_config['job_id']).fine_tuned_model

# Create output directory structure
run_name = run_config['run_name']
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Load all prompts we want to test
with open(TEST_PROMPTS, 'r') as file:
    prompts = json.load(file)

batch_job = []
print(f"Creating batch job for {run_name}. The batch will contain {len(prompts)} prompts. With {len(test_set)} pairs each.")
for task in prompts:
    title = task['title']
    prompt_template = task['prompt']
    prompt_id = task['id']

    for index, row in test_set.iterrows():
        product1, product2 = row['title_left'], row['title_right']
        label = row.get('label') 
        pair_id = row['pair_id']
        
        custom_id = f"{prompt_id};{pair_id};{label}"
        prompt = create_prompt(prompt_template, custom_id, fine_tune_model, product1, product2)
        batch_job.append(prompt)
        

# Save the input file
input_file_path = os.path.join(test_dir, f"{run_name}_input.jsonl")
with open(input_file_path, "w") as f:
    for request in batch_job:
        f.write(json.dumps(request) + "\n")
        
# upload the batch file to openai
batch_input_file = client.files.create(
    file=open(input_file_path, "rb"),
    purpose="batch"
)

# create the batch
batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={"description": run_name}
)

# Save the run configuration
run_config = {
    "run_name": run_name,
    "timestamp": timestamp,
    "model": fine_tune_model,
    "batch_id": batch.id,
    "input_file_id": batch_input_file.id,
    "endpoint": "/v1/chat/completions",
    "completion_window": "24h",
    "metadata": batch.metadata,
    "created_at": batch.created_at,
    "status": batch.status
}

with open(os.path.join(test_dir, "test_run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print(f"Run configuration and input files saved to: {test_dir}")
print(f"Batch ID: {batch.id}")


Creating batch job for fine-tune-wdc-small-regular. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809
Batch ID: batch_681cf7dfec08819091db0cd4b4499fe4


In [14]:
# Load the result file 
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809")

Batch status: completed
Created at: 2025-05-08 18:28:47
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809/batch_output_batch_681cf7dfec08819091db0cd4b4499fe4.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809/testing_stats.csv
Best performing task: 0.8207885304659498


'../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809/testing_stats.csv'

## Basic upsampeling

In [14]:
create_batch_job("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-basic-upsample/20250508_231510", TEST_SET_PATH, TEST_PROMPTS)

Creating batch job for fine-tune-wdc-basic-upsample. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-basic-upsample/20250508_231510
Batch ID: batch_681db606cfc88190b1009bc7e4dc780c


In [23]:
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-basic-upsample/20250508_231510")

Batch status: completed
Created at: 2025-05-09 08:00:06
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-basic-upsample/20250508_231510/batch_output_batch_681db606cfc88190b1009bc7e4dc780c.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-basic-upsample/20250508_231510/testing_stats.csv
Best performing task: 0.8007085916740478


'../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-basic-upsample/20250508_231510/testing_stats.csv'

## Simple swapping

In [13]:
create_batch_job("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-simple-swapping/20250508_204823", TEST_SET_PATH, TEST_PROMPTS)

Creating batch job for fine-tune-wdc-simple-swapping. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-simple-swapping/20250508_204823
Batch ID: batch_681d1d567d708190b581ea72151b8f2b


In [12]:
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-simple-swapping/20250508_204823")

Batch status: completed
Created at: 2025-05-08 21:08:38
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-simple-swapping/20250508_204823/batch_output_batch_681d1d567d708190b581ea72151b8f2b.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-simple-swapping/20250508_204823/testing_stats.csv
Best performing task: 0.7935702199661591


'../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-simple-swapping/20250508_204823/testing_stats.csv'

## WDC 25 swap finetuning

In [20]:
test_dir = "../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-swapping_0_25/20250507_155922"

# Load the test set
test_set = pd.read_pickle("/ceph/aasteine/fine-tuning-paper/data/wdc/wdcproducts80cc20rnd050un_test_gs.pkl")


# open fine-tune run config
with open(os.path.join(test_dir, "fine-tune-run_config.json"), "r") as f:
    run_config = json.load(f)
    
# get the name of the fine-tuned model via the openai api
fine_tune_model = client.fine_tuning.jobs.retrieve(run_config['job_id']).fine_tuned_model

# Create output directory structure
run_name = run_config['run_name']
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Load all prompts we want to test
with open(TEST_PROMPTS, 'r') as file:
    prompts = json.load(file)

batch_job = []
print(f"Creating batch job for {run_name}. The batch will contain {len(prompts)} prompts. With {len(test_set)} pairs each.")
for task in prompts:
    title = task['title']
    prompt_template = task['prompt']
    prompt_id = task['id']

    for index, row in test_set.iterrows():
        product1, product2 = row['title_left'], row['title_right']
        label = row.get('label') 
        pair_id = row['pair_id']
        
        custom_id = f"{prompt_id};{pair_id};{label}"
        prompt = create_prompt(prompt_template, custom_id, fine_tune_model, product1, product2)
        batch_job.append(prompt)
        

# Save the input file
input_file_path = os.path.join(test_dir, f"testing_{run_name}_input.jsonl")
with open(input_file_path, "w") as f:
    for request in batch_job:
        f.write(json.dumps(request) + "\n")
        
# upload the batch file to openai
batch_input_file = client.files.create(
    file=open(input_file_path, "rb"),
    purpose="batch"
)

# create the batch
batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={"description": run_name}
)

# Save the run configuration
run_config = {
    "run_name": run_name,
    "timestamp": timestamp,
    "model": fine_tune_model,
    "batch_id": batch.id,
    "input_file_id": batch_input_file.id,
    "endpoint": "/v1/chat/completions",
    "completion_window": "24h",
    "metadata": batch.metadata,
    "created_at": batch.created_at,
    "status": batch.status
}

with open(os.path.join(test_dir, "test_run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print(f"Run configuration and input files saved to: {test_dir}")
print(f"Batch ID: {batch.id}")


Creating batch job for fine-tune-wdc-swapping_0_25. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-swapping_0_25/20250507_155922
Batch ID: batch_681b7f43602c8190b032088afb2e963b


In [10]:
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-swapping_0_25/20250507_155922")

Batch status: completed
Created at: 2025-05-07 15:41:55
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-swapping_0_25/20250507_155922/batch_output_batch_681b7f43602c8190b032088afb2e963b.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-swapping_0_25/20250507_155922/testing_stats.csv
Best performing task: 0.825964252116651


'../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-swapping_0_25/20250507_155922/testing_stats.csv'

## 100% swapping

In [22]:
create_batch_job("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-100-swapping/20250509_103256", TEST_SET_PATH, TEST_PROMPTS)

Creating batch job for fine-tune-wdc-100-swapping. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-100-swapping/20250509_103256
Batch ID: batch_681dcce790608190bdfed2faf36e51c0


## Nplaug

In [11]:
create_batch_job("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-nplaug/20250508_204534", TEST_SET_PATH, TEST_PROMPTS)

Creating batch job for fine-tune-wdc-nplaug. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-nplaug/20250508_204534
Batch ID: batch_681d1ce7df848190beeac76b611eea00


In [10]:
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-nplaug/20250508_204534")

Batch status: completed
Created at: 2025-05-08 21:06:47
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-nplaug/20250508_204534/batch_output_batch_681d1ce7df848190beeac76b611eea00.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-nplaug/20250508_204534/testing_stats.csv
Best performing task: 0.7814029363784666


'../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-nplaug/20250508_204534/testing_stats.csv'

## Randomized swapping

In [12]:
create_batch_job("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-random-swapping/20250508_212306", TEST_SET_PATH, TEST_PROMPTS)

Creating batch job for fine-tune-wdc-random-swapping. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-random-swapping/20250508_212306
Batch ID: batch_681d1d2818d881908afa0782b30104ca


In [11]:
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-random-swapping/20250508_212306")

Batch status: completed
Created at: 2025-05-08 21:07:52
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-random-swapping/20250508_212306/batch_output_batch_681d1d2818d881908afa0782b30104ca.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-random-swapping/20250508_212306/testing_stats.csv
Best performing task: 0.7615262321144675


'../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-random-swapping/20250508_212306/testing_stats.csv'

## Aggreagate scores

In [20]:
def aggregate_testing_stats(base_dir):
    # Find all testing_stats.csv files
    stats_files = glob.glob(os.path.join(base_dir, "**", "testing_stats.csv"), recursive=True)
    
    results = []
    
    for file_path in stats_files:
        # Extract run name from path
        # Path format: .../run-name/date_time/testing_stats.csv
        path_parts = file_path.split(os.sep)
        run_name = path_parts[-3]  # Get the run name from the path
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Find the row with the best F1 score
        best_result = df.loc[df['f1_score'].idxmax()]
        
        # Create result dictionary
        result = {
            'run_name': run_name,
            'task': best_result['task'],
            'accuracy': best_result['accuracy'],
            'f1_score': best_result['f1_score'],
            'precision': best_result['precision'],
            'recall': best_result['recall']
        }
        
        results.append(result)
    
    # Create DataFrame and save to CSV
    results_df = pd.DataFrame(results)
    output_path = os.path.join(base_dir, 'testing_overview.csv')
    results_df.to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")



In [21]:
base_dir = "../../results/gpt-4.1-mini-2025-04-14"
aggregate_testing_stats(base_dir)

Results saved to ../../results/gpt-4.1-mini-2025-04-14/testing_overview.csv


## Get Scores

In [66]:
model = "ft:gpt-4o-mini-2024-07-18:wbsg-uni-mannheim:explanations:9rAaVb9c"

In [2]:
full_datasets = [
    {"dataset_name": "wdc-fullsize", "dataset_path": "../data/wdc/wdcproducts80cc20rnd050un_test_gs.pkl"},
    {"dataset_name": "abt-buy-full", "dataset_path": "../data/abt-buy/abt-buy-gs.pkl"}, 
    {"dataset_name": "amazon-google-full", "dataset_path": "../data/amazon-google/amazon-google-gs.pkl"},
    {"dataset_name": "dblp-acm", "dataset_path": "../data/dblp-acm/dblp-acm-gs.pkl"},
    {"dataset_name": "dblp-scholar", "dataset_path": "../data/dblp-scholar/dblp-scholar-gs.pkl"},
    {"dataset_name": "walmart-amazon", "dataset_path": "../data/walmart-amazon/walmart-amazon-gs.pkl"}
    
]

In [9]:
for model in models:
    batch_job = []

    for dataset in full_datasets:
        # Load the dataset
        df = pd.read_pickle(dataset["dataset_path"])

        # Load all prompts we want to test
        with open('../prompts/domain_promts.json', 'r') as file:
            prompts = json.load(file)

        result_rows = []

        for task in prompts:
            title = task['title']
            prompt_template = task['prompt']

            for index, row in df.iterrows():
                if "dblp" in dataset["dataset_name"]:
                    product1 = f"{row['title_left']}; {row['authors_left']}; {row['venue_left']}; {row['year_left']}"
                    product2=f"{row['title_right']}; {row['authors_right']}; {row['venue_right']}; {row['year_right']}"
                else:
                    product1, product2 = row['title_left'], row['title_right']
                    
                label = row.get('label') 
                
                custom_id = f"{dataset['dataset_name']};{title};{row['pair_id']};{label}"
                prompt = create_prompt(prompt_template, custom_id, model, product1, product2)
                batch_job.append(prompt)
                
    print(len(batch_job))
    print(batch_job[0])

    batch_file_path = "dblp_filter.jsonl"
    with open(batch_file_path, "w") as f:
        for request in batch_job:
            f.write(json.dumps(request) + "\n")

    batch_input_file = client.files.create(
        file=open(batch_file_path, "rb"),
        purpose="batch"
    )

    batch_input_file_id = batch_input_file.id

    batch = client.batches.create(
        input_file_id=batch_input_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"description": "Test scholar datasets"}
    )

    # delete the batch input file
    os.remove(batch_file_path)



34836
{'custom_id': 'wdc-fullsize;domain-complex-free (Product);61830419#18905357;0', 'method': 'POST', 'url': '/v1/chat/completions', 'body': {'model': 'ft:gpt-4o-mini-2024-07-18:wbsg-uni-mannheim::A1xT61am', 'messages': [{'role': 'user', 'content': 'Do the two product descriptions refer to the same real-world product? Entity 1: MultiPlus C 12/2000/80-30. Entity 2: DDR4 16GB 3200 Kingston Fury Black.'}], 'max_tokens': 5, 'temperature': 0}}
34836
{'custom_id': 'wdc-fullsize;domain-complex-free (Product);61830419#18905357;0', 'method': 'POST', 'url': '/v1/chat/completions', 'body': {'model': 'ft:gpt-4o-mini-2024-07-18:wbsg-uni-mannheim::A1yQPMEC', 'messages': [{'role': 'user', 'content': 'Do the two product descriptions refer to the same real-world product? Entity 1: MultiPlus C 12/2000/80-30. Entity 2: DDR4 16GB 3200 Kingston Fury Black.'}], 'max_tokens': 5, 'temperature': 0}}


In [3]:
# Dictionary to hold the DataFrames
dataframes = {}

# Load each dataset into a DataFrame and store in the dictionary
for dataset in full_datasets:
    dataset_name = dataset["dataset_name"]
    dataset_path = dataset["dataset_path"]
    try:
        df = pd.read_pickle(dataset_path)
        dataframes[dataset_name] = df
        print(f"Loaded {dataset_name} successfully.")
    except Exception as e:
        print(f"Failed to load {dataset_name} from {dataset_path}. Error: {e}")
        
# Function to lookup label in the original dataframes using pair_id
def lookup_label(row):
    dataset_name = row['dataset']
    pair_id = row['pair_id']
    if dataset_name in dataframes:
        original_df = dataframes[dataset_name]
        # Assuming pair_id is a unique identifier in the original dataframe
        if pair_id in original_df['pair_id'].values:
            return original_df.loc[original_df['pair_id'] == pair_id, 'label'].values[0]
    return None

Loaded wdc-fullsize successfully.
Loaded abt-buy-full successfully.
Loaded amazon-google-full successfully.
Loaded dblp-acm successfully.
Loaded dblp-scholar successfully.
Loaded walmart-amazon successfully.


In [10]:
## Download the results
batch_list = client.batches.list(limit=19)

for batch_job in batch_list.data:
    # convert the unix timestamp to a human-readable format
    created_at = datetime.utcfromtimestamp(batch_job.created_at).strftime('%Y-%m-%d %H:%M:%S')
    output_file_id = batch_job.output_file_id
    # Ensure the batch has completed
    if output_file_id:
        # Step 3: Download the output file content
        file_content = client.files.content(output_file_id)
        
        # Step 4: Write the content to a .jsonl file
        with open(f"../results/gpt-4o-mini/tobedetermined/{output_file_id}.jsonl", "w") as file:
            file.write(file_content.text)
        
        print("Batch results saved to batch_output.jsonl")
    else:
        print("Batch is not completed or output file is not available.")
    print(batch_job.id, batch_job.status, created_at, batch_job.output_file_id)

Batch results saved to batch_output.jsonl
batch_JpeFAtrkdejXKiyE3lwJVhEe completed 2024-08-30 18:12:49 file-d2eciGLVxRisn6e7Iw2Br3Vh
Batch results saved to batch_output.jsonl
batch_08eGvNHcm23hrJTuQE06IolA completed 2024-08-30 18:12:45 file-5MgFICNPuAyx4fqyVFrGTOuT
Batch results saved to batch_output.jsonl
batch_tZaBH7fYilJlPJgtVnRAMvLh completed 2024-08-30 18:12:35 file-jS8neFCTD78m95q01chFbwJs
Batch results saved to batch_output.jsonl
batch_8Tr18iUYwIzMjygI4VQuxLgB completed 2024-08-30 18:12:30 file-qv6nYdtjyBsFlW2w3akOPqPN
Batch results saved to batch_output.jsonl
batch_3YcqKwIINI9lwROscaUMCWEt completed 2024-08-30 17:01:58 file-uJ1fE2HCryhhXSpDKm4tz2Cl
Batch results saved to batch_output.jsonl
batch_JGuBNsYYvTYRtnvDX6StATqs completed 2024-08-30 17:01:53 file-Sff5nykjAtoiFK9C1NEwpqTu
Batch results saved to batch_output.jsonl
batch_n4wPLxaMXPQ7loOPmuWWRuZI completed 2024-08-30 17:01:48 file-S1NOw4jmLN1SVAw0Zv1NYenl
Batch results saved to batch_output.jsonl
batch_6AZrEkx4LTVKgij81t5JG